# Wake convergence plotting example

`run_wake_convergence.ipynb`が生成した`wake_convergence.csv`を読み込み、

- `NumWakeNodes`固定時の`WakeNumIter`収束
- `WakeNumIter`固定時の`NumWakeNodes`収束

を同じ形式で比較する。

## 1. importとCSV読込

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator, MaxNLocator, ScalarFormatter
import numpy as np
import pandas as pd
from IPython.display import display

output_dir = Path.cwd().resolve()
results_path = output_dir / "wake_convergence.wbhv.csv"

if not results_path.exists():
    raise FileNotFoundError(results_path)

df = pd.read_csv(results_path)

if "passed" in df.columns:
    passed_mask = (
        df["passed"]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes"})
    )
    df = df.loc[passed_mask].copy()

if df.empty:
    raise ValueError("No passed wake-convergence cases were found.")

for flag_column in ("in_wake_iter_sweep", "in_wake_node_sweep"):
    if flag_column not in df.columns:
        raise KeyError(
            f"Missing {flag_column}. Regenerate the CSV with "
            "run_wake_convergence.ipynb."
        )
    df[flag_column] = (
        df[flag_column]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes"})
    )

iter_df = df.loc[df["in_wake_iter_sweep"]].sort_values("wake_iter").copy()
node_df = df.loc[df["in_wake_node_sweep"]].sort_values("wake_nodes").copy()

if iter_df.empty or node_df.empty:
    raise ValueError("Both wake sweeps must contain at least one passed case.")

print("WakeNumIter sweep")
display(iter_df[["wake_iter", "wake_nodes", "elapsed_s"]])
print("NumWakeNodes sweep")
display(node_df[["wake_iter", "wake_nodes", "elapsed_s"]])

## 2. 機体軸横力係数 $C_Y$

VSPAEROの$C_D$と$C_S$から、基準横滑り角における機体軸横力係数を作る。

In [ ]:
beta0 = (
    np.deg2rad(df["Beta_deg"].to_numpy(dtype=float))
    if "Beta_deg" in df.columns
    else np.zeros(len(df))
)

df["CY_Base"] = (
    -df["CD_Base"] * np.sin(beta0)
    + df["CS_Base"] * np.cos(beta0)
)

for suffix in (
    "U",
    "Alpha",
    "p",
    "q",
    "r",
    "Mach",
    "delta_e",
    "delta_a",
    "delta_r",
):
    cd_column = f"CD_{suffix}"
    cs_column = f"CS_{suffix}"
    if cd_column in df.columns and cs_column in df.columns:
        df[f"CY_{suffix}"] = (
            -df[cd_column] * np.sin(beta0)
            + df[cs_column] * np.cos(beta0)
        )

if {"CD_Base", "CD_Beta", "CS_Base", "CS_Beta"}.issubset(df.columns):
    df["CY_Beta"] = (
        -df["CD_Base"] * np.cos(beta0)
        -df["CD_Beta"] * np.sin(beta0)
        +df["CS_Beta"] * np.cos(beta0)
        -df["CS_Base"] * np.sin(beta0)
    )

iter_df = df.loc[df["in_wake_iter_sweep"]].sort_values("wake_iter").copy()
node_df = df.loc[df["in_wake_node_sweep"]].sort_values("wake_nodes").copy()

## 3. 共通描画処理

同じ係数群を2つの収束試験で描くため、1列の描画とpanel gridの組立てだけを共通化する。

In [ ]:
Y_RANGE_OVERRIDES = {}

def plot_column(ax, data, x_column: str, column: str, ylabel: str) -> None:
    ax.set_ylabel(ylabel)

    if column not in data.columns:
        ax.text(
            0.5,
            0.5,
            f"missing\n{column}",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        return

    values = pd.to_numeric(data[column], errors="coerce")
    ax.plot(
        data[x_column],
        values,
        marker="o",
        linewidth=1.5,
        markerfacecolor="none",
    )

    if column in Y_RANGE_OVERRIDES:
        y_min, y_max = Y_RANGE_OVERRIDES[column]
    else:
        finite_values = values[np.isfinite(values.to_numpy(dtype=float))]
        if finite_values.empty:
            y_min, y_max = -1.0, 1.0
        else:
            y_min = min(float(finite_values.min()), 0.0)
            y_max = max(float(finite_values.max()), 0.0)
            span = y_max - y_min
            if span == 0.0:
                span = max(abs(y_min), 1.0) * 0.1
            padding = 0.08 * span
            y_min -= padding
            y_max += padding

    ax.set_ylim(y_min, y_max)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))

    formatter = ScalarFormatter(useMathText=True)
    formatter.set_powerlimits((-3, 3))
    ax.yaxis.set_major_formatter(formatter)
    ax.grid(True, which="major", linewidth=0.5)
    ax.grid(True, which="minor", linewidth=0.3, alpha=0.4)
    ax.ticklabel_format(axis="y", style="plain", useOffset=False)

def plot_grid(
    data,
    x_column: str,
    x_label: str,
    panels,
    nrows: int,
    ncols: int,
    figsize,
    output_path: Path,
) -> None:
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=figsize,
        sharex=True,
        constrained_layout=True,
    )
    axes = np.asarray(axes).reshape(nrows, ncols)

    for ax, (column, ylabel) in zip(axes.flat, panels):
        plot_column(ax, data, x_column, column, ylabel)

    for ax in axes[-1, :]:
        ax.set_xlabel(x_label)

    # fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.show()

## 4. Base空力係数

In [ ]:
base_panels = [
    ("CL_Base", r"$C_L$"),
    ("CMl_Base", r"$C_l$"),
    ("CY_Base", r"$C_Y$"),
    ("CMm_Base", r"$C_m$"),
    ("CD_Base", r"$C_D$"),
    ("CMn_Base", r"$C_n$"),
]

studies = [
    (
        "wake_iter",
        iter_df,
        "wake_iter",
        r"Wake iteration $N_{\mathrm{iter}}$",
    ),
    (
        "wake_nodes",
        node_df,
        "wake_nodes",
        r"Wake nodes $N_{\mathrm{nodes}}$",
    ),
]

for tag, data, x_column, x_label in studies:
    plot_grid(
        data,
        x_column,
        x_label,
        base_panels,
        3,
        2,
        (9, 8),
        output_dir / f"{tag}_base.png",
    )

## 5. 安定微係数

In [ ]:
derivative_rows = [
    ("U", r"U"),
    ("Alpha", r"\alpha"),
    ("Beta", r"\beta"),
    ("p", r"\hat p"),
    ("q", r"\hat q"),
    ("r", r"\hat r"),
    ("delta_e", r"\delta_e"),
    ("delta_a", r"\delta_a"),
    ("delta_r", r"\delta_r"),
]

longitudinal_columns = [
    ("CL", r"C_L"),
    ("CD", r"C_D"),
    ("CMm", r"C_m"),
]
lateral_columns = [
    ("CY", r"C_Y"),
    ("CMl", r"C_l"),
    ("CMn", r"C_n"),
]

longitudinal_panels = [
    (
        f"{prefix}_{suffix}",
        rf"$\partial {coefficient_label}/\partial {variable_label}$",
    )
    for suffix, variable_label in derivative_rows
    for prefix, coefficient_label in longitudinal_columns
]
lateral_panels = [
    (
        f"{prefix}_{suffix}",
        rf"$\partial {coefficient_label}/\partial {variable_label}$",
    )
    for suffix, variable_label in derivative_rows
    for prefix, coefficient_label in lateral_columns
]

for tag, data, x_column, x_label in studies:
    plot_grid(
        data,
        x_column,
        x_label,
        longitudinal_panels,
        len(derivative_rows),
        len(longitudinal_columns),
        (12, 20),
        output_dir / f"{tag}_longitudinal_derivatives.png",
    )
    plot_grid(
        data,
        x_column,
        x_label,
        lateral_panels,
        len(derivative_rows),
        len(lateral_columns),
        (12, 20),
        output_dir / f"{tag}_lateral_derivatives.png",
    )

## 6. 計算時間

In [ ]:
for tag, data, x_column, x_label in studies:
    elapsed = data[[x_column, "vspaero_sweep_elapsed_s"]].dropna()
    if elapsed.empty:
        print(f"{tag}: elapsed-time plot skipped.")
        continue

    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    ax.plot(
        elapsed[x_column],
        elapsed["vspaero_sweep_elapsed_s"],
        marker="o",
        markerfacecolor="none",
    )
    ax.set_xlabel(x_label)
    ax.set_ylabel("VSPAERO sweep elapsed time [s]")
    ax.grid(True)
    # fig.savefig(
    #     output_dir / f"vspaero_sweep_elapsed_s_vs_{tag}.png",
    #     dpi=150,
    #     bbox_inches="tight",
    # )
    plt.show()